### Dataset selection and description

This project explores the **Heart Disease dataset** from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/45/heart+disease), using the processed Cleveland data.

The selected dataset contains **303 patient records and 14 columns: 13 features and one target variable**. Although the full database contains 76 attributes, this processed dataset uses the commonly studied subset of 14.

Important variables include age, sex, chest pain type (`cp`), resting blood pressure (`trestbps`), serum cholesterol (`chol`), and maximum heart rate achieved (`thalach`). The target variable, `num`, records heart disease status: **0 indicates absence, while values 1–4 indicate presence**.

I selected this dataset to explore patient characteristics and their relationships with heart disease status. It supports numerical summaries, category counts, grouped comparisons, and correlation analysis.

Missing values occur in `ca` (four records) and `thal` (two records). The original data contains no duplicate rows. Adding the derived `gender` column increases the notebook's column count from 14 to 15.

In [2]:
# Import the libraries used to load, inspect, and analyse the heart disease dataset.
# Fetch the UCI features and target, then load the complete CSV into df for exploration.
from ucimlrepo import fetch_ucirepo
import requests
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import io
import requests
import seaborn as sns
from scipy.stats import norm
import math
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from IPython.display import HTML, display
import plotly.express as px
from scipy.stats import probplot
%matplotlib inline
from pathlib import Path
import sqlite3 

pd.set_option('display.max_columns', None)
  
# fetch dataset 
heart_disease = fetch_ucirepo(id=45) 
  
# data (as pandas dataframes) 
heart_disease_df = heart_disease.data.features 
y = heart_disease.data.targets 

df_url = 'https://archive.ics.uci.edu/static/public/45/data.csv'
url_content = requests.get(df_url, verify=False).content
df = pd.read_csv(io.StringIO(url_content.decode('utf-8')))
# metadata 
#print(heart_disease.metadata) 
display(heart_disease_df.head())

# variable information 
display(type(heart_disease))

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0


ucimlrepo.dotdict.dotdict

In [3]:
# Preview of the first five records to check the imported data and column layout.
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [4]:
# Translate the sex codes into readable labels in a new gender column.
# Keep the original sex column for reference.
sexes_dict = {
    1: "male",
    0: "female"
}

df['gender'] = df['sex'].replace(sexes_dict)

In [5]:
# Count the records labelled male and female to inspect the sample composition.
df['gender'].value_counts()

gender
male      206
female     97
Name: count, dtype: int64

In [6]:
# Compare numeric column averages between the two sex groups.
df.groupby('gender').mean(numeric_only=True) 

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
gender,,,,,,,,,,,,,,
female,55.721649,0.0,3.144330,133.340206,261.752577,0.123711,0.958763,151.226804,0.226804,0.867010,1.567010,0.546392,3.656250,0.536082
male,53.834951,1.0,3.165049,130.912621,239.601942,0.160194,1.004854,148.844660,0.373786,1.120874,1.616505,0.732673,5.239024,1.126214


In [7]:
# Find the largest observed value in each numeric column to inspect upper ranges.
df.max(numeric_only=True) 

age          77.0
sex           1.0
cp            4.0
trestbps    200.0
chol        564.0
fbs           1.0
restecg       2.0
thalach     202.0
exang         1.0
oldpeak       6.2
slope         3.0
ca            3.0
thal          7.0
num           4.0
dtype: float64

In [8]:
# Find the smallest observed value in each numeric column to inspect lower ranges.
df.min(numeric_only=True) 

age          29.0
sex           0.0
cp            1.0
trestbps     94.0
chol        126.0
fbs           0.0
restecg       0.0
thalach      71.0
exang         0.0
oldpeak       0.0
slope         1.0
ca            0.0
thal          3.0
num           0.0
dtype: float64

In [9]:
# Count records for each heart disease target code.
# Code 0 indicates absence; codes 1 through 4 indicate presence.
df['num'].value_counts()

num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64

In [10]:
# Preview the first five records again, including the newly added gender labels.
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,gender
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0,male
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2,male
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1,male
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0,male
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0,female


In [11]:
# Inspect the last five records to check the end of the dataset.
df.tail()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,gender
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0,1,male
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0,2,male
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0,3,male
301,57,0,2,130,236,0,2,174,0,0.0,2,1.0,3.0,1,female
302,38,1,3,138,175,0,0,173,0,0.0,1,NaN,3.0,0,male


In [12]:
# Show the number of rows and columns, including the derived gender column.
df.shape

(303, 15)

In [13]:
# List the column names available for subsequent analysis.
df.columns

Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num', 'gender'],
      dtype='str')

In [14]:
# Inspect column data types, non-missing counts, and memory usage.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
 14  gender    303 non-null    object 
dtypes: float64(3), int64(11), object(1)
memory usage: 35.6+ KB


In [15]:
# Summarise numeric columns using counts, means, standard deviations, and quartiles.
df.describe()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,299.000000,301.000000,303.000000
mean,54.438944,0.679868,3.158416,131.689769,246.693069,0.148515,0.990099,149.607261,0.326733,1.039604,1.600660,0.672241,4.734219,0.937294
std,9.038662,0.467299,0.960126,17.599748,51.776918,0.356198,0.994971,22.875003,0.469794,1.161075,0.616226,0.937438,1.939706,1.228536
min,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,1.000000,0.000000,3.000000,0.000000
25%,48.000000,0.000000,3.000000,120.000000,211.000000,0.000000,0.000000,133.500000,0.000000,0.000000,1.000000,0.000000,3.000000,0.000000
50%,56.000000,1.000000,3.000000,130.000000,241.000000,0.000000,1.000000,153.000000,0.000000,0.800000,2.000000,0.000000,3.000000,0.000000
75%,61.000000,1.000000,4.000000,140.000000,275.000000,0.000000,2.000000,166.000000,1.000000,1.600000,2.000000,1.000000,7.000000,2.000000
max,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,3.000000,7.000000,4.000000


In [16]:
# Check each column storage type to distinguish numeric and text data.
df.dtypes

age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca          float64
thal        float64
num           int64
gender       object
dtype: object

In [17]:
# Count distinct non-missing values in each column to inspect its variety.
df.nunique()

age          41
sex           2
cp            4
trestbps     50
chol        152
fbs           2
restecg       3
thalach      91
exang         2
oldpeak      40
slope         3
ca            4
thal          3
num           5
gender        2
dtype: int64

In [18]:
# Inspect ten randomly selected records beyond the first and last rows.
df.sample(10)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,gender
245,67,1,4,120,237,0,0,71,0,1.0,2,0.0,3.0,2,male
13,44,1,2,120,263,0,0,173,0,0.0,1,0.0,7.0,0,male
227,67,0,3,152,277,0,0,172,0,0.0,1,1.0,3.0,0,female
67,54,1,3,150,232,0,2,165,0,1.6,1,0.0,7.0,0,male
123,55,1,4,140,217,0,0,111,1,5.6,3,0.0,7.0,3,male
7,57,0,4,120,354,0,0,163,1,0.6,1,0.0,3.0,0,female
236,56,1,4,130,283,1,2,103,1,1.6,3,0.0,7.0,2,male
128,44,1,2,120,220,0,0,170,0,0.0,1,0.0,3.0,0,male
39,61,1,3,150,243,1,0,137,1,1.0,2,0.0,3.0,0,male
85,44,1,3,140,235,0,2,180,0,0.0,1,0.0,3.0,0,male


In [19]:
# Count missing values in each column to identify incomplete variables.
df.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
num         0
gender      0
dtype: int64

In [20]:
# Express each column missing-value count as a percentage of all records.
round(((df.isnull().sum() / len(df)) * 100), 2)

age         0.00
sex         0.00
cp          0.00
trestbps    0.00
chol        0.00
fbs         0.00
restecg     0.00
thalach     0.00
exang       0.00
oldpeak     0.00
slope       0.00
ca          1.32
thal        0.66
num         0.00
gender      0.00
dtype: float64

In [21]:
# Count exact duplicate rows beyond their first occurrence.
df.duplicated().sum() 

np.int64(0)

In [22]:
# Calculate column averages for numeric data, excluding missing values.
df.mean(numeric_only=True)

age          54.438944
sex           0.679868
cp            3.158416
trestbps    131.689769
chol        246.693069
fbs           0.148515
restecg       0.990099
thalach     149.607261
exang         0.326733
oldpeak       1.039604
slope         1.600660
ca            0.672241
thal          4.734219
num           0.937294
dtype: float64

In [23]:
# Count patients in each chest pain category.
# Codes: 1 = typical angina, 2 = atypical angina,
# 3 = non-anginal pain, and 4 = asymptomatic.
df['cp'].value_counts()

cp
4    144
3     86
2     50
1     23
Name: count, dtype: int64

In [24]:
# Create a separate copy and remove rows with any missing values.
# Display the complete-case dataset while keeping the original df unchanged.
heart_df = df.copy()
heart_df = heart_df.dropna()
heart_df

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,gender
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0,male
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2,male
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1,male
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0,male
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0,female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,57,0,4,140,241,0,0,123,1,0.2,2,0.0,7.0,1,female
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0,1,male
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0,2,male
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0,3,male


In [25]:
# Inspect original records missing ca, the number of major vessels.
df.loc[df['ca'].isna()]

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,gender
166,52,1,3,138,223,0,0,169,0,0.0,1,NaN,3.0,0,male
192,43,1,4,132,247,1,2,143,1,0.1,2,NaN,7.0,1,male
287,58,1,2,125,220,0,0,144,0,0.4,2,NaN,7.0,0,male
302,38,1,3,138,175,0,0,173,0,0.0,1,NaN,3.0,0,male


In [26]:
# Inspect original records missing the thal test result.
df.loc[df['thal'].isna()]

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,gender
87,53,0,3,128,216,0,2,115,0,0.0,1,0.0,NaN,0,female
266,52,1,4,128,204,1,0,156,1,1.0,2,0.0,NaN,2,male


In [27]:
# Calculate and display Pearson correlations between numeric columns in df.
# Each pair uses records with both values present; df retains its missing values.
# Interpret coded categories cautiously, and do not infer causation from correlation.
corr_matrix = df.corr(numeric_only=True)
display(corr_matrix)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
age,1.000000,-0.097542,0.104139,0.284946,0.208950,0.118530,0.148868,-0.393806,0.091661,0.203805,0.161770,0.362605,0.127389,0.222853
sex,-0.097542,1.000000,0.010084,-0.064456,-0.199915,0.047862,0.021647,-0.048663,0.146201,0.102173,0.037533,0.093185,0.380936,0.224469
cp,0.104139,0.010084,1.000000,-0.036077,0.072319,-0.039975,0.067505,-0.334422,0.384060,0.202277,0.152050,0.233214,0.265246,0.407075
trestbps,0.284946,-0.064456,-0.036077,1.000000,0.130120,0.175340,0.146560,-0.045351,0.064762,0.189171,0.117382,0.098773,0.133554,0.157754
chol,0.208950,-0.199915,0.072319,0.130120,1.000000,0.009841,0.171043,-0.003432,0.061310,0.046564,-0.004062,0.119000,0.014214,0.070909
fbs,0.118530,0.047862,-0.039975,0.175340,0.009841,1.000000,0.069564,-0.007854,0.025665,0.005747,0.059894,0.145478,0.071358,0.059186
restecg,0.148868,0.021647,0.067505,0.146560,0.171043,0.069564,1.000000,-0.083389,0.084867,0.114133,0.133946,0.128343,0.024531,0.183696
thalach,-0.393806,-0.048663,-0.334422,-0.045351,-0.003432,-0.007854,-0.083389,1.000000,-0.378103,-0.343085,-0.385601,-0.264246,-0.279631,-0.415040
exang,0.091661,0.146201,0.384060,0.064762,0.061310,0.025665,0.084867,-0.378103,1.000000,0.288223,0.257748,0.145570,0.329680,0.397057
oldpeak,0.203805,0.102173,0.202277,0.189171,0.046564,0.005747,0.114133,-0.343085,0.288223,1.000000,0.577537,0.295832,0.341004,0.504092


In [28]:

heart_df = heart_df.rename(columns={'num': 'goal'})

In [29]:
heart_df['patient_id'] = heart_df.index + 1
new_order = ['patient_id', 'age', 'gender', 'sex', 'cp', 'trestbps', 'chol','fbs','restecg','thalach','exang','oldpeak', 'slope', 'ca','thal','goal']
heart_df = heart_df[new_order]

In [30]:
heart_df.head()

,patient_id,age,gender,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,goal
0,1,63,male,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,2,67,male,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,3,67,male,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,4,37,male,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,5,41,female,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [31]:
heart_df.info()

<class 'pandas.DataFrame'>
Index: 297 entries, 0 to 301
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   patient_id  297 non-null    int64  
 1   age         297 non-null    int64  
 2   gender      297 non-null    object 
 3   sex         297 non-null    int64  
 4   cp          297 non-null    int64  
 5   trestbps    297 non-null    int64  
 6   chol        297 non-null    int64  
 7   fbs         297 non-null    int64  
 8   restecg     297 non-null    int64  
 9   thalach     297 non-null    int64  
 10  exang       297 non-null    int64  
 11  oldpeak     297 non-null    float64
 12  slope       297 non-null    int64  
 13  ca          297 non-null    float64
 14  thal        297 non-null    float64
 15  goal        297 non-null    int64  
dtypes: float64(3), int64(12), object(1)
memory usage: 39.4+ KB


In [32]:
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

csv_path = data_dir /  "cleaned_dataset.csv"
heart_df.to_csv(csv_path, index=False)

In [33]:
database_dir = Path("database")
database_dir.mkdir(parents=True, exist_ok=True)
db_path = database_dir / "project.db"


create_sql = """
CREATE TABLE IF NOT EXISTS patients (
    patient_id INTEGER PRIMARY KEY,
    age INTEGER NOT NULL,
    gender TEXT NOT NULL,
    sex INTEGER NOT NULL,
    cp INTEGER NOT NULL,
    trestbps INTEGER NOT NULL,
    chol INTEGER NOT NULL,
    fbs INTEGER NOT NULL,
    restecg INTEGER NOT NULL,
    thalach INTEGER NOT NULL,
    exang INTEGER NOT NULL,
    oldpeak FLOAT NOT NULL,
    slope INTEGER NOT NULL,
    ca FLOAT NOT NULL,
    thal FLOAT NOT NULL,
    goal INTEGER 
)
"""
with sqlite3.connect(db_path) as conn:
    conn.execute(create_sql)

In [34]:
upload_df = pd.read_csv(csv_path)
upload_df.to_sql("patients", conn, if_exists="replace", index=False)
conn.close()
print("Data uploaded successfully.")

Data uploaded successfully.


In [35]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT * FROM patients").fetchall()

[(1, 63, 'male', 1, 1, 145, 233, 1, 2, 150, 0, 2.3, 3, 0.0, 6.0, 0),
 (2, 67, 'male', 1, 4, 160, 286, 0, 2, 108, 1, 1.5, 2, 3.0, 3.0, 2),
 (3, 67, 'male', 1, 4, 120, 229, 0, 2, 129, 1, 2.6, 2, 2.0, 7.0, 1),
 (4, 37, 'male', 1, 3, 130, 250, 0, 0, 187, 0, 3.5, 3, 0.0, 3.0, 0),
 (5, 41, 'female', 0, 2, 130, 204, 0, 2, 172, 0, 1.4, 1, 0.0, 3.0, 0),
 (6, 56, 'male', 1, 2, 120, 236, 0, 0, 178, 0, 0.8, 1, 0.0, 3.0, 0),
 (7, 62, 'female', 0, 4, 140, 268, 0, 2, 160, 0, 3.6, 3, 2.0, 3.0, 3),
 (8, 57, 'female', 0, 4, 120, 354, 0, 0, 163, 1, 0.6, 1, 0.0, 3.0, 0),
 (9, 63, 'male', 1, 4, 130, 254, 0, 2, 147, 0, 1.4, 2, 1.0, 7.0, 2),
 (10, 53, 'male', 1, 4, 140, 203, 1, 2, 155, 1, 3.1, 3, 0.0, 7.0, 1),
 (11, 57, 'male', 1, 4, 140, 192, 0, 0, 148, 0, 0.4, 2, 0.0, 6.0, 0),
 (12, 56, 'female', 0, 2, 140, 294, 0, 2, 153, 0, 1.3, 2, 0.0, 3.0, 0),
 (13, 56, 'male', 1, 3, 130, 256, 1, 2, 142, 1, 0.6, 2, 1.0, 6.0, 2),
 (14, 44, 'male', 1, 2, 120, 263, 0, 0, 173, 0, 0.0, 1, 0.0, 7.0, 0),
 (15, 52, 'male', 1, 